## Setup


In [ ]:
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """find pyproject.toml"""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"No pyproject.toml found above {start}")


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("sys.path[0]:", sys.path[0])


Project root: /Users/thatt/Dev/AI project/data-morph
sys.path[0]: /Users/thatt/Dev/AI project/data-morph


In [2]:
import json
import time

from src.extractor import CSVExtractor, JSONExtractor

CSV_FIXTURES = PROJECT_ROOT / "tests" / "extractor" / "fixtures"
JSON_FIXTURES = CSV_FIXTURES / "json"

print("CSV fixtures:", CSV_FIXTURES)
print("JSON fixtures:", JSON_FIXTURES)


CSV fixtures: /Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures
JSON fixtures: /Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json


In [3]:
def show_fixture(path: Path, max_bytes: int = 800) -> None:
    """Print the raw fixture file. Truncate large files so the notebook stays readable."""
    size = path.stat().st_size
    print(f"--- {path.name}  ({size:,} bytes) ---")
    if size == 0:
        print("<empty file>")
        return
    raw = path.read_bytes()
    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = raw.decode("latin-1", errors="replace")
    if len(text) > max_bytes:
        print(text[:max_bytes])
        print(f"... [truncated {len(text) - max_bytes:,} more chars]")
    else:
        print(text)


def summarize_warnings(envelope: dict) -> None:
    """One line per warning: `[severity] CODE`."""
    warnings = envelope.get("warnings", [])
    if not warnings:
        print("warnings: (none)")
        return
    print(f"warnings: {len(warnings)}")
    for w in warnings:
        print(f"  [{w['severity']:5}] {w['code']}")


def run(extractor, path: Path, *, show_full_envelope: bool = True) -> dict:
    """Run an extractor on a fixture and pretty-print everything."""
    show_fixture(path)
    start = time.monotonic()
    env = extractor.extract(path)
    elapsed = time.monotonic() - start
    print()
    print(f"--- envelope (elapsed: {elapsed * 1000:.1f} ms) ---")
    if show_full_envelope:
        print(json.dumps(env, indent=2, default=str, ensure_ascii=False))
    else:
        skinny = {k: v for k, v in env.items() if k != "samples" and k != "schema"}
        skinny["schema"] = {
            k: (f"<{len(v)} paths>" if k == "paths" else v)
            for k, v in env["schema"].items()
        }
        skinny["samples"] = f"<elided ({len(env.get('samples') or {})} buckets)>"
        print(json.dumps(skinny, indent=2, default=str))
    print()
    summarize_warnings(env)
    return env


csv_ex = CSVExtractor()
json_ex = JSONExtractor()
print("Extractors instantiated.")

Extractors instantiated.


## 1.1 simple_users.csv


In [4]:
_ = run(csv_ex, CSV_FIXTURES / "simple_users.csv")

--- simple_users.csv  (140 bytes) ---
first_name,last_name,city,country
Alice,Smith,New York,USA
Bob,Jones,London,UK
Carol,Lee,Tokyo,JP
David,Park,Seoul,KR
Emma,Garcia,Madrid,ES


--- envelope (elapsed: 9.0 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/simple_users.csv",
  "file_size_bytes": 140,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": true,
    "row_count": 5,
    "columns": [
      {
        "name": "first_name",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          "Alice",
          "Bob",
          "Carol"
        ],
        "max_length": 5
      },
      {
        "name": "last_name",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          "Smith",
          "Jones",
          "Lee"
        ],
        "max_l

## 1.2 repeating_entity.csv


In [5]:
_ = run(csv_ex, CSV_FIXTURES / "repeating_entity.csv")

--- repeating_entity.csv  (294 bytes) ---
user_name,user_email,order_id,order_item,order_price
Alice Smith,alice@example.com,1001,Widget A,25.00
Alice Smith,alice@example.com,1002,Gadget B,14.50
Alice Smith,alice@example.com,1003,Tool C,7.25
Bob Jones,bob@example.com,1004,Widget A,25.00
Bob Jones,bob@example.com,1005,Sprocket D,49.99


--- envelope (elapsed: 3.5 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/repeating_entity.csv",
  "file_size_bytes": 294,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": true,
    "row_count": 5,
    "columns": [
      {
        "name": "user_name",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 2,
        "sample_values": [
          "Alice Smith",
          "Bob Jones"
        ],
        "max_length": 11
      },
      {
        "name": "user_email",
        "dtype": "string",
  

## 1.3 numeric_columns.csv


In [6]:
_ = run(csv_ex, CSV_FIXTURES / "numeric_columns.csv")

--- numeric_columns.csv  (78 bytes) ---
id,score,price
1,85.5,9.99
2,72.0,14.50
3,91.5,7.25
4,88.0,12.99
5,95.5,19.99


--- envelope (elapsed: 3.2 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/numeric_columns.csv",
  "file_size_bytes": 78,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": true,
    "row_count": 5,
    "columns": [
      {
        "name": "id",
        "dtype": "integer",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          1,
          2,
          3
        ],
        "min": 1,
        "max": 5
      },
      {
        "name": "score",
        "dtype": "float",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          85.5,
          72.0,
          91.5
        ],
        "min": 72.0,
        "max": 95.5
      },
      {
        "name": "price",
        "dtyp

## 1.4 mixed_dtype.csv


In [7]:
_ = run(csv_ex, CSV_FIXTURES / "mixed_dtype.csv")

--- mixed_dtype.csv  (39 bytes) ---
id,value
1,100
2,200
3,abc
4,400
5,xyz


--- envelope (elapsed: 2.9 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/mixed_dtype.csv",
  "file_size_bytes": 39,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": true,
    "row_count": 5,
    "columns": [
      {
        "name": "id",
        "dtype": "integer",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          1,
          2,
          3
        ],
        "min": 1,
        "max": 5
      },
      {
        "name": "value",
        "dtype": "mixed",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          "100",
          "200",
          "abc"
        ]
      }
    ]
  },
  "samples": {
    "head": [
      {
        "id": "1",
        "value": "100"
      },
      {
        "id": "2",
       

## 1.5 headerless.csv


In [8]:
_ = run(csv_ex, CSV_FIXTURES / "headerless.csv")

--- headerless.csv  (82 bytes) ---
1,Alice,New York
2,Bob,Los Angeles
3,Carol,Chicago
4,David,Houston
5,Emma,Phoenix


--- envelope (elapsed: 3.0 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/headerless.csv",
  "file_size_bytes": 82,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": false,
    "row_count": 5,
    "columns": [
      {
        "name": "c0",
        "dtype": "integer",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          1,
          2,
          3
        ],
        "min": 1,
        "max": 5
      },
      {
        "name": "c1",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          "Alice",
          "Bob",
          "Carol"
        ],
        "max_length": 5
      },
      {
        "name": "c2",
        "dtype": "string",
      

## 1.6 duplicate_columns.csv


In [9]:
_ = run(csv_ex, CSV_FIXTURES / "duplicate_columns.csv")

--- duplicate_columns.csv  (100 bytes) ---
name,name,email
Alice,Smith,alice@example.com
Bob,Jones,bob@example.com
Carol,Lee,carol@example.com


--- envelope (elapsed: 3.0 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/duplicate_columns.csv",
  "file_size_bytes": 100,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": true,
    "row_count": 3,
    "columns": [
      {
        "name": "name",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 3,
        "sample_values": [
          "Alice",
          "Bob",
          "Carol"
        ],
        "max_length": 5
      },
      {
        "name": "name.1",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 3,
        "sample_values": [
          "Smith",
          "Jones",
          "Lee"
        ],
        "max_length": 5
      },
      {
        "nam

## 1.7 empty_file.csv


In [10]:
_ = run(csv_ex, CSV_FIXTURES / "empty_file.csv")

--- empty_file.csv  (0 bytes) ---
<empty file>

--- envelope (elapsed: 0.4 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/empty_file.csv",
  "file_size_bytes": 0,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": false,
    "row_count": 0,
    "columns": []
  },
  "samples": {
    "head": [],
    "middle": [],
    "tail": []
  },
  "warnings": [
    {
      "code": "EMPTY_FILE",
      "severity": "error",
      "message": "File has zero data rows.",
      "context": {
        "row_count": 0
      }
    }
  ]
}

warnings: 1
  [error] EMPTY_FILE


## 1.8 large_file.csv, performance check (10 000 rows)


In [11]:
_ = run(csv_ex, CSV_FIXTURES / "large_file.csv", show_full_envelope=False)

--- large_file.csv  (191,509 bytes) ---
id,value,category
1,1.5,cat_1
2,3.0,cat_2
3,4.5,cat_3
4,6.0,cat_4
5,7.5,cat_5
6,9.0,cat_6
7,10.5,cat_0
8,12.0,cat_1
9,13.5,cat_2
10,15.0,cat_3
11,16.5,cat_4
12,18.0,cat_5
13,19.5,cat_6
14,21.0,cat_0
15,22.5,cat_1
16,24.0,cat_2
17,25.5,cat_3
18,27.0,cat_4
19,28.5,cat_5
20,30.0,cat_6
21,31.5,cat_0
22,33.0,cat_1
23,34.5,cat_2
24,36.0,cat_3
25,37.5,cat_4
26,39.0,cat_5
27,40.5,cat_6
28,42.0,cat_0
29,43.5,cat_1
30,45.0,cat_2
31,46.5,cat_3
32,48.0,cat_4
33,49.5,cat_5
34,51.0,cat_6
35,52.5,cat_0
36,54.0,cat_1
37,55.5,cat_2
38,57.0,cat_3
39,58.5,cat_4
40,60.0,cat_5
41,61.5,cat_6
42,63.0,cat_0
43,64.5,cat_1
44,66.0,cat_2
45,67.5,cat_3
46,69.0,cat_4
47,70.5,cat_5
48,72.0,cat_6
49,73.5,cat_0
50,75.0,cat_1
51,76.5,cat_2
52,78.0,cat_3
53,79.5,cat_4
5
... [truncated 190,709 more chars]

--- envelope (elapsed: 31.6 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/large_file.csv",
  "file_size_bytes": 19

---

# Part 2 JSONExtractor


## 2.1 simple_records.json


In [12]:
_ = run(json_ex, JSON_FIXTURES / "simple_records.json")

--- simple_records.json  (298 bytes) ---
[
  {"id": 1, "name": "Alice", "email": "alice@example.com"},
  {"id": 2, "name": "Bob",   "email": "bob@example.com"},
  {"id": 3, "name": "Carol", "email": "carol@example.com"},
  {"id": 4, "name": "Dave",  "email": "dave@example.com"},
  {"id": 5, "name": "Eve",   "email": "eve@example.com"}
]


--- envelope (elapsed: 0.3 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/simple_records.json",
  "file_size_bytes": 298,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "record_array",
    "max_depth": 1,
    "paths": [
      {
        "path": "[].id",
        "dtype": "integer",
        "presence": 1.0,
        "sample_values": [
          1,
          2,
          3
        ],
        "unique_count": 3,
        "min": 1,
        "max": 5
      },
      {
        "path": "[].email",
        "dtype": "string",
        "presence": 1.0,
        "sample

## 2.2 nested_root.json


In [13]:
_ = run(json_ex, JSON_FIXTURES / "nested_root.json")

--- nested_root.json  (135 bytes) ---
{
  "meta": {"version": 1, "generator": "data-morph"},
  "users": [
    {"id": 1, "name": "Alice"},
    {"id": 2, "name": "Bob"}
  ]
}


--- envelope (elapsed: 0.4 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/nested_root.json",
  "file_size_bytes": 135,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "object",
    "max_depth": 3,
    "paths": [
      {
        "path": "meta",
        "dtype": "object",
        "presence": 1.0
      },
      {
        "path": "meta.version",
        "dtype": "integer",
        "presence": 1.0,
        "sample_values": [
          1
        ],
        "unique_count": 1,
        "min": 1,
        "max": 1
      },
      {
        "path": "meta.generator",
        "dtype": "string",
        "presence": 1.0,
        "sample_values": [
          "data-morph"
        ],
        "unique_count": 1,
        "max_length": 1

## 2.3 optional_keys.json


In [14]:
_ = run(json_ex, JSON_FIXTURES / "optional_keys.json")

--- optional_keys.json  (211 bytes) ---
[
  {"id": 1, "name": "Alice", "phone": "555-0001"},
  {"id": 2, "name": "Bob"},
  {"id": 3, "name": "Carol", "phone": "555-0003"},
  {"id": 4, "name": "Dave"},
  {"id": 5, "name": "Eve", "phone": "555-0005"}
]


--- envelope (elapsed: 0.4 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/optional_keys.json",
  "file_size_bytes": 211,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "record_array",
    "max_depth": 1,
    "paths": [
      {
        "path": "[].id",
        "dtype": "integer",
        "presence": 1.0,
        "sample_values": [
          1,
          2,
          3
        ],
        "unique_count": 3,
        "min": 1,
        "max": 5
      },
      {
        "path": "[].phone",
        "dtype": "string",
        "presence": 0.6,
        "sample_values": [
          "555-0001",
          "555-0003",
          "555-0005"
        ],
 

## 2.4 mixed_type_path.json


In [15]:
_ = run(json_ex, JSON_FIXTURES / "mixed_type_path.json")

--- mixed_type_path.json  (95 bytes) ---
[
  {"id": 1, "name": "Alice"},
  {"id": "two", "name": "Bob"},
  {"id": 3, "name": "Carol"}
]


--- envelope (elapsed: 0.3 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/mixed_type_path.json",
  "file_size_bytes": 95,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "record_array",
    "max_depth": 1,
    "paths": [
      {
        "path": "[].id",
        "dtype": "mixed",
        "presence": 1.0,
        "sample_values": [
          1,
          "two",
          3
        ],
        "unique_count": 3
      },
      {
        "path": "[].name",
        "dtype": "string",
        "presence": 1.0,
        "sample_values": [
          "Alice",
          "Bob",
          "Carol"
        ],
        "unique_count": 3,
        "max_length": 5
      }
    ],
    "root_array_length": 3
  },
  "samples": {
    "head": [
      {
        "id": 1,
        "

## 2.5 deeply_nested.json


In [16]:
_ = run(json_ex, JSON_FIXTURES / "deeply_nested.json")

--- deeply_nested.json  (49 bytes) ---
{"a":{"b":{"c":{"d":{"e":{"f":{"g":"deep"}}}}}}}


--- envelope (elapsed: 0.3 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/deeply_nested.json",
  "file_size_bytes": 49,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "object",
    "max_depth": 7,
    "paths": [
      {
        "path": "a",
        "dtype": "object",
        "presence": 1.0
      },
      {
        "path": "a.b",
        "dtype": "object",
        "presence": 1.0
      },
      {
        "path": "a.b.c",
        "dtype": "object",
        "presence": 1.0
      },
      {
        "path": "a.b.c.d",
        "dtype": "object",
        "presence": 1.0
      },
      {
        "path": "a.b.c.d.e",
        "dtype": "object",
        "presence": 1.0
      },
      {
        "path": "a.b.c.d.e.f",
        "dtype": "object",
        "presence": 1.0
      },
      {
        "path": "a.b.c.

## 2.6 heterogeneous_array.json


In [17]:
_ = run(json_ex, JSON_FIXTURES / "heterogeneous_array.json")

--- heterogeneous_array.json  (110 bytes) ---
{
  "items": [
    {"alpha": 1, "beta": 2},
    {"gamma": 3, "delta": 4},
    {"epsilon": 5, "zeta": 6}
  ]
}


--- envelope (elapsed: 0.3 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/heterogeneous_array.json",
  "file_size_bytes": 110,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "object",
    "max_depth": 3,
    "paths": [
      {
        "path": "items",
        "dtype": "array",
        "presence": 1.0
      },
      {
        "path": "items[].alpha",
        "dtype": "integer",
        "presence": 1.0,
        "sample_values": [
          1
        ],
        "unique_count": 1,
        "min": 1,
        "max": 1
      },
      {
        "path": "items[].beta",
        "dtype": "integer",
        "presence": 1.0,
        "sample_values": [
          2
        ],
        "unique_count": 1,
        "min": 2,
        "max": 2
      },

## 2.7 dates.json


In [18]:
_ = run(json_ex, JSON_FIXTURES / "dates.json")

--- dates.json  (126 bytes) ---
[
  {"id": 1, "created_at": "2026-05-20"},
  {"id": 2, "created_at": "2026-05-21"},
  {"id": 3, "created_at": "2026-05-22"}
]


--- envelope (elapsed: 0.2 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/dates.json",
  "file_size_bytes": 126,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "record_array",
    "max_depth": 1,
    "paths": [
      {
        "path": "[].id",
        "dtype": "integer",
        "presence": 1.0,
        "sample_values": [
          1,
          2,
          3
        ],
        "unique_count": 3,
        "min": 1,
        "max": 3
      },
      {
        "path": "[].created_at",
        "dtype": "string",
        "presence": 1.0,
        "sample_values": [
          "2026-05-20",
          "2026-05-21",
          "2026-05-22"
        ],
        "unique_count": 3,
        "max_length": 10
      }
    ],
    "root_array_length":

## 2.8 empty.json


In [19]:
_ = run(json_ex, JSON_FIXTURES / "empty.json")

--- empty.json  (0 bytes) ---
<empty file>

--- envelope (elapsed: 0.2 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/empty.json",
  "file_size_bytes": 0,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "unsupported",
    "paths": []
  },
  "samples": {},
  "warnings": [
    {
      "code": "EMPTY_FILE",
      "severity": "error",
      "message": "File is byte-empty.",
      "context": {
        "file_size_bytes": 0
      }
    }
  ]
}

warnings: 1
  [error] EMPTY_FILE


## 2.9 malformed.json


In [20]:
_ = run(json_ex, JSON_FIXTURES / "malformed.json")

--- malformed.json  (8 bytes) ---
{"a": 1,

--- envelope (elapsed: 0.2 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/malformed.json",
  "file_size_bytes": 8,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "unsupported",
    "paths": []
  },
  "samples": {},
  "warnings": [
    {
      "code": "MALFORMED_JSON",
      "severity": "error",
      "message": "JSON parse failed: Expecting property name enclosed in double quotes: line 1 column 9 (char 8)",
      "context": {
        "error": "Expecting property name enclosed in double quotes: line 1 column 9 (char 8)",
        "line": 1,
        "column": 9
      }
    }
  ]
}

warnings: 1
  [error] MALFORMED_JSON


## 2.10 scalar_root.json


In [21]:
_ = run(json_ex, JSON_FIXTURES / "scalar_root.json")

--- scalar_root.json  (28 bytes) ---
"just a string at the root"


--- envelope (elapsed: 0.1 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/scalar_root.json",
  "file_size_bytes": 28,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "unsupported",
    "paths": []
  },
  "samples": {},
  "warnings": [
    {
      "code": "ROOT_SHAPE_UNSUPPORTED",
      "severity": "error",
      "message": "JSON root is str; Phase 2 supports only object roots and record-array roots (≥80% object elements).",
      "context": {
        "observed_root_type": "str"
      }
    }
  ]
}

warnings: 1
  [error] ROOT_SHAPE_UNSUPPORTED


## 2.11 large_array.json


In [22]:
_ = run(json_ex, JSON_FIXTURES / "large_array.json", show_full_envelope=False)

--- large_array.json  (3,366,670 bytes) ---
[{"id": 0, "name": "user_0", "email": "u0@example.com"}, {"id": 1, "name": "user_1", "email": "u1@example.com"}, {"id": 2, "name": "user_2", "email": "u2@example.com"}, {"id": 3, "name": "user_3", "email": "u3@example.com"}, {"id": 4, "name": "user_4", "email": "u4@example.com"}, {"id": 5, "name": "user_5", "email": "u5@example.com"}, {"id": 6, "name": "user_6", "email": "u6@example.com"}, {"id": 7, "name": "user_7", "email": "u7@example.com"}, {"id": 8, "name": "user_8", "email": "u8@example.com"}, {"id": 9, "name": "user_9", "email": "u9@example.com"}, {"id": 10, "name": "user_10", "email": "u10@example.com"}, {"id": 11, "name": "user_11", "email": "u11@example.com"}, {"id": 12, "name": "user_12", "email": "u12@example.com"}, {"id": 13, "name": "user_13", "email": "u13@example.com"}, {"i
... [truncated 3,365,870 more chars]

--- envelope (elapsed: 95.9 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/test

## 2.12 (bonus)


In [23]:
tight = JSONExtractor(max_file_size_bytes=1)
_ = run(tight, JSON_FIXTURES / "simple_records.json", show_full_envelope=True)

--- simple_records.json  (298 bytes) ---
[
  {"id": 1, "name": "Alice", "email": "alice@example.com"},
  {"id": 2, "name": "Bob",   "email": "bob@example.com"},
  {"id": 3, "name": "Carol", "email": "carol@example.com"},
  {"id": 4, "name": "Dave",  "email": "dave@example.com"},
  {"id": 5, "name": "Eve",   "email": "eve@example.com"}
]


--- envelope (elapsed: 0.1 ms) ---
{
  "format": "json",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/json/simple_records.json",
  "file_size_bytes": 298,
  "encoding": "utf-8",
  "schema_version": "0.1",
  "schema": {
    "root_shape": "unsupported",
    "paths": []
  },
  "samples": {},
  "warnings": [
    {
      "code": "FILE_TOO_LARGE",
      "severity": "error",
      "message": "File size 298 bytes exceeds cap 1 bytes.",
      "context": {
        "file_size_bytes": 298,
        "max_file_size_bytes": 1
      }
    }
  ]
}

warnings: 1
  [error] FILE_TOO_LARGE


In [24]:
MY_FILE = (
    PROJECT_ROOT / "tests" / "extractor" / "fixtures" / "simple_users.csv"
)  # ← change me

suffix = MY_FILE.suffix.lower()
if suffix == ".csv":
    ex = csv_ex
elif suffix == ".json":
    ex = json_ex
else:
    raise ValueError(f"No extractor for {suffix!r} (TXT extractor lands in W3 Phase 3)")

_ = run(ex, MY_FILE)

--- simple_users.csv  (140 bytes) ---
first_name,last_name,city,country
Alice,Smith,New York,USA
Bob,Jones,London,UK
Carol,Lee,Tokyo,JP
David,Park,Seoul,KR
Emma,Garcia,Madrid,ES


--- envelope (elapsed: 3.3 ms) ---
{
  "format": "csv",
  "file_path": "/Users/thatt/Dev/AI project/data-morph/tests/extractor/fixtures/simple_users.csv",
  "file_size_bytes": 140,
  "encoding": "utf-8-sig",
  "schema_version": "0.1",
  "schema": {
    "delimiter": ",",
    "quote_char": "\"",
    "has_header": true,
    "row_count": 5,
    "columns": [
      {
        "name": "first_name",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          "Alice",
          "Bob",
          "Carol"
        ],
        "max_length": 5
      },
      {
        "name": "last_name",
        "dtype": "string",
        "null_count": 0,
        "unique_count": 5,
        "sample_values": [
          "Smith",
          "Jones",
          "Lee"
        ],
        "max_l